In [1]:
"""
==============================================================================
🚀 [Step 3] Letterbox Normalization & BBox Precision Clipping
==============================================================================
### @Author       : 한의정 (Data Engineering Lead)
### @Description  : 원본 이미지의 종횡비(Aspect Ratio) 왜곡을 방지하기 위한 
###                 Letterbox 리사이징 및 모델 입력 규격(800x800) 표준화.
###                 이미지 경계선 밖으로 이탈하는 BBox 좌표에 대한 정밀 클리핑 로직 구현.
### @Key Metric   : Resolution 800x800 (Padding: 114, 114, 114), Zero-Data-Loss Clipping
==============================================================================
"""

'\n==============================================================================\n🚀 [Step 3] Letterbox Normalization & BBox Precision Clipping\n==============================================================================\n### @Author       : 한의정 (Data Engineering Lead)\n### @Description  : 원본 이미지의 종횡비(Aspect Ratio) 왜곡을 방지하기 위한 \n###                 Letterbox 리사이징 및 모델 입력 규격(800x800) 표준화.\n###                 이미지 경계선 밖으로 이탈하는 BBox 좌표에 대한 정밀 클리핑 로직 구현.\n### @Key Metric   : Resolution 800x800 (Padding: 114, 114, 114), Zero-Data-Loss Clipping\n==============================================================================\n'

In [2]:
# ============================================================
# [Cell 0] 환경 설정 — Colab / 로컬 자동 감지
# ============================================================
import sys, os

try:
    is_colab = 'google.colab' in str(get_ipython())
except NameError:
    is_colab = False

if is_colab:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_DIR = '/content/pill_detection_project'
    if not os.path.exists(REPO_DIR):
        os.system('git clone https://github.com/wina0901/pill_detection_project.git ' + REPO_DIR)

    sys.path.insert(0, REPO_DIR)
    BASE_DIR = '/content/drive/MyDrive/data/초급_프로젝트/dataset'

else:
    sys.path.insert(0, os.path.abspath('..'))
    BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '../data'))

print(f"✅ 환경: {'Colab' if is_colab else '로컬'}")
print(f"✅ PROJECT: {sys.path[0]}")
print(f"✅ DATA:    {BASE_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 환경: Colab
✅ PROJECT: /content/pill_detection_project
✅ DATA:    /content/drive/MyDrive/data/초급_프로젝트/dataset


In [3]:
############################################################
# 1. [Data Pipeline Phase 4] 
#     Letterbox 리사이징 및 모델 입력 규격화 (Standardization)
############################################################
import cv2
import json
import os
import numpy as np
import glob
from tqdm.auto import tqdm
from collections import defaultdict
import warnings
import platform

warnings.filterwarnings('ignore')

# ==========================================
# 1. 설정 및 경로 (Configuration)
# ==========================================
# local에서 작업 시 BASE_DIR을 자동으로 감지하도록 개선
#  - 플랫폼 감지 로직을 추가하여 자동으로 BASE_DIR을 설정하도록 개선

TARGET_SIZE = 800  # YOLO, Faster R-CNN 등 대부분의 모델이 선호하는 고해상도 규격

# [Input] 증강이 완료된 Train 데이터와 분리된 Val 데이터
AUG_JSON_PATH = os.path.join(BASE_DIR, 'train_augmented_final.json')
VAL_JSON_PATH = os.path.join(BASE_DIR, 'val.json')

# [Output] 모델이 바로 소화할 수 있는 최종 데이터 경로
LB_TRAIN_IMG_DIR = os.path.join(BASE_DIR, 'letterbox_images/train') 
LB_VAL_IMG_DIR   = os.path.join(BASE_DIR, 'letterbox_images/val')   
LB_TRAIN_JSON    = os.path.join(BASE_DIR, 'train_letterbox.json')
LB_VAL_JSON      = os.path.join(BASE_DIR, 'val_letterbox.json')

os.makedirs(LB_TRAIN_IMG_DIR, exist_ok=True)
os.makedirs(LB_VAL_IMG_DIR,   exist_ok=True)

# ==========================================
# 2. 글로벌 파일 인덱싱 (I/O 병목 최적화)
# ==========================================
print("⚡ [최적화] 전체 이미지 파일 경로를 메모리에 캐싱합니다...")
# 함수 외부에서 단 1번만 실행하여 디스크 I/O 부하를 극단적으로 줄입니다.
all_files = glob.glob(os.path.join(BASE_DIR, "**", "*.png"), recursive=True) + \
            glob.glob(os.path.join(BASE_DIR, "**", "*.jpg"), recursive=True) + \
            glob.glob(os.path.join(BASE_DIR, "**", "*.JPG"), recursive=True)
GLOBAL_PATH_MAP = {os.path.basename(f): f for f in all_files}
print(f"[OK] 총 {len(GLOBAL_PATH_MAP):,}개의 파일 경로 인덱싱 완료.\n")

# ==========================================
# 3. Letterbox 변환 핵심 엔진
# ==========================================
def letterbox_with_bbox(image, bboxes, target_size=800):
    """
    이미지의 원본 비율(Aspect Ratio)을 유지한 채 target_size로 리사이즈하고,
    남는 공간을 회색(114, 114, 114) 패딩으로 채웁니다. BBox 좌표도 동기화됩니다.
    """
    h, w = image.shape[:2]
    
    # 1) 가로/세로 중 긴 쪽을 기준으로 스케일(Scale)을 결정
    scale = target_size / max(h, w)
    new_w, new_h = int(w * scale), int(h * scale)
    
    # 2) 비율을 유지하며 이미지 리사이즈
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    # 3) 캔버스 정중앙 배치를 위한 패딩(Padding) 여백 계산
    pad_top  = (target_size - new_h) // 2
    pad_left = (target_size - new_w) // 2

    # 4) 패딩 적용 (114, 114, 114는 ImageNet 평균 픽셀값과 유사하여 CNN 모델에 최적화됨)
    padded = cv2.copyMakeBorder(
        resized,
        pad_top,  target_size - new_h - pad_top,
        pad_left, target_size - new_w - pad_left,
        cv2.BORDER_CONSTANT, value=(114, 114, 114)
    )

    # 5) BBox 좌표 선형 변환 (x_min, y_min, width, height)
    new_bboxes = []
    for x, y, bw, bh in bboxes:
        nx  = x * scale + pad_left
        ny  = y * scale + pad_top
        nbw = bw * scale
        nbh = bh * scale
        
        # 클리핑(Clipping): 계산 오차로 인해 캔버스를 벗어나는 것을 방지
        nx = max(0.0, min(float(target_size), nx))
        ny = max(0.0, min(float(target_size), ny))
        nbw = min(target_size - nx, nbw)
        nbh = min(target_size - ny, nbh)

        # 유효성 검사 (크기가 0이 되어버린 박스 폐기)
        if nbw > 1.0 and nbh > 1.0:
            new_bboxes.append([round(nx, 2), round(ny, 2), round(nbw, 2), round(nbh, 2)])
        else:
            new_bboxes.append(None) 

    return padded, new_bboxes

# ==========================================
# 4. 데이터 가공 파이프라인
# ==========================================
def process_pipeline(json_path, out_json_path, img_dir, desc):
    """지정된 COCO JSON을 읽어 전체 이미지를 Letterbox 규격으로 일괄 변환합니다."""
    
    if not os.path.exists(json_path):
        print(f"🚨 [에러] 파일을 찾을 수 없습니다: {json_path}")
        return

    with open(json_path, 'r', encoding='utf-8') as f:
        coco = json.load(f)

    # O(1) 매칭을 위한 Annotation 그룹화 해시맵
    id_to_anns = defaultdict(list)
    for ann in coco['annotations']:
        id_to_anns[ann['image_id']].append(ann)

    new_images, new_annotations = [], []
    skipped_bbox = 0 

    for img_info in tqdm(coco['images'], desc=desc):
        f_name = os.path.basename(img_info['file_name'])
        
        if f_name not in GLOBAL_PATH_MAP:
            continue

        # OpenCV 한글 경로 호환 디코딩
        img_array = np.fromfile(GLOBAL_PATH_MAP[f_name], np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
        
        if img is None: continue

        anns   = id_to_anns.get(img_info['id'], [])
        bboxes = [ann['bbox'] for ann in anns]

        # 🚀 핵심 엔진 호출
        lb_img, transformed_bboxes = letterbox_with_bbox(img, bboxes, TARGET_SIZE)

        # 1) 규격화된 이미지 디스크 저장
        save_name = f"lb_{img_info['id']:06d}.jpg"
        save_path = os.path.join(img_dir, save_name)
        result, enc = cv2.imencode('.jpg', lb_img)
        if result:
            with open(save_path, 'w+b') as f:
                enc.tofile(f)

        # 2) JSON Image 메타데이터 업데이트
        new_images.append({
            'id':        img_info['id'],
            'file_name': save_name,
            'width':     TARGET_SIZE,
            'height':    TARGET_SIZE
        })

        # 3) JSON Annotation 업데이트 (1:1 매칭 보장)
        for ann, final_bbox in zip(anns, transformed_bboxes):
            if final_bbox is None:
                skipped_bbox += 1
                continue
            
            # 얕은 복사본을 만들어 원본 훼손 방지
            new_ann = ann.copy()
            new_ann['bbox'] = final_bbox
            new_ann['area'] = round(final_bbox[2] * final_bbox[3], 2)
            new_annotations.append(new_ann)

    # 4) 최종 JSON 직렬화
    with open(out_json_path, 'w', encoding='utf-8') as f:
        json.dump({
            'images':      new_images,
            'annotations': new_annotations,
            'categories':  coco['categories']
        }, f, ensure_ascii=False) # indent 제거로 속도 최적화

    print(f"✅ [저장 완료] {os.path.basename(out_json_path)}")
    print(f"   ▶ 이미지 {len(new_images):,}장 / 객체 {len(new_annotations):,}개")
    if skipped_bbox > 0:
        print(f"   ▶ 🗑️ 유효하지 않은 BBox {skipped_bbox:,}개 안전하게 필터링됨\n")

# ==========================================
# 5. 파이프라인 실행
# ==========================================
print(f"📦 [Phase 4] Letterbox 데이터 가공 파이프라인을 시작합니다.\n")

process_pipeline(AUG_JSON_PATH, LB_TRAIN_JSON, LB_TRAIN_IMG_DIR, "Train Letterbox 변환")
process_pipeline(VAL_JSON_PATH, LB_VAL_JSON,   LB_VAL_IMG_DIR,   "Val Letterbox 변환")

print("✨ [엔지니어링 완료] 모델링 투입 준비가 모두 끝났습니다!")
print(f"  - 🎯 Train 정답지: {os.path.basename(LB_TRAIN_JSON)}")
print(f"  - 🧪 Val 정답지  : {os.path.basename(LB_VAL_JSON)}")

⚡ [최적화] 전체 이미지 파일 경로를 메모리에 캐싱합니다...
[OK] 총 3,976개의 파일 경로 인덱싱 완료.

📦 [Phase 4] Letterbox 데이터 가공 파이프라인을 시작합니다.



Train Letterbox 변환:   0%|          | 0/1792 [00:00<?, ?it/s]

✅ [저장 완료] train_letterbox.json
   ▶ 이미지 1,792장 / 객체 6,196개
   ▶ 🗑️ 유효하지 않은 BBox 3개 안전하게 필터링됨



Val Letterbox 변환:   0%|          | 0/139 [00:00<?, ?it/s]

✅ [저장 완료] val_letterbox.json
   ▶ 이미지 139장 / 객체 431개
✨ [엔지니어링 완료] 모델링 투입 준비가 모두 끝났습니다!
  - 🎯 Train 정답지: train_letterbox.json
  - 🧪 Val 정답지  : val_letterbox.json
